# **Universal benchmark — four GPT-2 routing architectures**

Mohammad Al Dridi

Every model is rebuilt here, loaded from its trained weights, and scored by
**identical code on identical inputs**. Nothing is retrained.

| | Model | Owner | Trained on |
|---|---|---|---|
| 1 | MoE top-1, 4 experts | Tamara | alpaca-cleaned |
| 2 | MoE soft routing (NAM-inspired) | Tamara | alpaca |
| 3 | MoE + category-conditioned router | Tamara | alpaca-cleaned |
| 4 | AAG, 8 chunks x 4 options | Mohammad | alpaca |

A pretrained GPT-2 is included as a reference line. It was never
instruction-tuned, so it should score worst -- if it doesn't, something is
wrong with the harness rather than with the models.

Runtime -> Change runtime type -> **L4 GPU**. Roughly 20-30 minutes total.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import copy
import json
import math
import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, GPT2LMHeadModel
from transformers.activations import ACT2FN
from datasets import load_dataset
from safetensors.torch import load_file
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1b. Mount Drive

Mount before anything touches a Drive path -- writing to `/content/drive`
while unmounted creates plain local folders that then block the real mount.

In [ ]:
import shutil
from google.colab import drive

if os.path.exists('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    shutil.rmtree('/content/drive')
    print('cleared a stray /content/drive')

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive did not mount'
print('Drive mounted')

## 2. Config

In [ ]:
CKPT = '/content/drive/MyDrive/ece1508_checkpoints'
OUT  = '/content/drive/MyDrive/ece1508_results/benchmark'
os.makedirs(OUT, exist_ok=True)

SEED         = 42
MAX_LENGTH   = 512
EVAL_BATCH   = 8
N_EVAL       = 800     # held-out examples per eval set
GEN_TOKENS   = 60

torch.manual_seed(SEED)

MODELS = {
    'gpt2-dense': dict(
        label='GPT-2 (no fine-tuning)', owner='reference', kind='dense',
        ckpt=None, trained_on='--'),
    'moe-top1': dict(
        label='MoE top-1 (4 experts)', owner='Tamara', kind='moe', routing='top1',
        ckpt=f'{CKPT}/gpt2-moe-alpacaclean-aux', trained_on='alpaca-cleaned'),
    'moe-soft': dict(
        label='MoE soft routing (NAM)', owner='Tamara', kind='moe', routing='soft',
        ckpt=f'{CKPT}/gpt2-moe-alpaca-aux-nam', trained_on='alpaca'),
    'moe-cond': dict(
        label='MoE + category conditioning', owner='Tamara', kind='moe-cond',
        ckpt=f'{CKPT}/gpt2-moe-alpacaclean-aux-cond', trained_on='alpaca-cleaned'),
    'aag': dict(
        label='AAG (8 chunks x 4 options)', owner='Mohammad', kind='aag',
        num_chunks=8, num_options=4,
        ckpt=f'{CKPT}/gp2-aag-alpaca', trained_on='alpaca'),
}

for name, spec in MODELS.items():
    if spec['ckpt'] is None:
        print(f'{name:12s} pretrained GPT-2, no checkpoint')
        continue
    path = f"{spec['ckpt']}/model.safetensors"
    if os.path.exists(path):
        print(f'{name:12s} FOUND   {os.path.getsize(path) / 1e9:5.2f} GB')
    else:
        print(f'{name:12s} MISSING {path}')

## 3. Architectures

Each model's layer is reimplemented here with **parameter names matching its
checkpoint exactly**, so weights load without renaming. The loader below
refuses to continue if any key fails to match -- scoring a partly-random model
and reporting it as someone's trained result would be worse than useless.

### Tamara — MoE (top-1 and soft routing)

In [ ]:
class GPT2MoELayer(nn.Module):
    """Router plus num_experts deepcopies of the pretrained MLP."""

    def __init__(self, original_mlp, num_experts=4, routing='top1'):
        super().__init__()
        self.num_experts = num_experts
        self.routing = routing
        hidden_dim = original_mlp.c_fc.weight.shape[0]
        self.router = nn.Linear(hidden_dim, num_experts)
        self.experts = nn.ModuleList(
            [copy.deepcopy(original_mlp) for _ in range(num_experts)])
        self.routing_log = []
        self.track_routing = False

    def forward(self, hidden_states, *args, **kwargs):
        shape = hidden_states.shape
        flat = hidden_states.view(-1, shape[-1])
        probs = torch.softmax(self.router(flat), dim=-1)
        top1 = probs.argmax(dim=-1)
        if self.track_routing:
            self.routing_log.append(top1.detach().cpu())

        if self.routing == 'soft':
            out = torch.zeros_like(flat)
            for e in range(self.num_experts):
                out += self.experts[e](flat) * probs[:, e].unsqueeze(-1)
        else:
            weights = probs.gather(1, top1.unsqueeze(-1))
            out = None
            for e in range(self.num_experts):
                mask = top1 == e
                if not mask.any():
                    continue
                y = self.experts[e](flat[mask]) * weights[mask]
                if out is None:
                    out = torch.zeros(flat.shape, dtype=y.dtype, device=y.device)
                out[mask] = y
            if out is None:
                out = torch.zeros_like(flat)

        return out.view(shape)

    def stored_params(self):
        return sum(p.numel() for p in self.experts.parameters())

    def active_params(self):
        per = sum(p.numel() for p in self.experts[0].parameters())
        return per * (self.num_experts if self.routing == 'soft' else 1)

    def n_options(self):
        return self.num_experts

    def configurations(self):
        return float(self.num_experts)

### Tamara — category-conditioned router

A 7-category embedding table (4 dims) is concatenated onto the hidden state
before the router, so the router sees `hidden_dim + 4` inputs. During training
the embedding is dropped to zeros 20% of the time; **at inference her code
always uses the zero vector**, so no category labels are needed here and every
model sees byte-identical inputs.

In [ ]:
class GPT2MoECondLayer(nn.Module):
    """MoE whose router is conditioned on a task-category embedding."""

    def __init__(self, original_mlp, num_experts=4, num_categories=7, cat_embed_dim=4):
        super().__init__()
        self.num_experts = num_experts
        self.cat_embed_dim = cat_embed_dim
        hidden_dim = original_mlp.c_fc.weight.shape[0]
        self.category_embeddings = nn.Embedding(num_categories, cat_embed_dim)
        self.router = nn.Linear(hidden_dim + cat_embed_dim, num_experts)
        self.experts = nn.ModuleList(
            [copy.deepcopy(original_mlp) for _ in range(num_experts)])
        self.routing_log = []
        self.track_routing = False

    def forward(self, hidden_states, *args, **kwargs):
        shape = hidden_states.shape
        flat = hidden_states.view(-1, shape[-1])

        # Inference path from her notebook: zero category vector.
        cat = torch.zeros(shape[0], shape[1], self.cat_embed_dim,
                          device=hidden_states.device, dtype=hidden_states.dtype)
        conditioned = torch.cat([hidden_states, cat], dim=-1)
        flat_cond = conditioned.view(-1, shape[-1] + self.cat_embed_dim)

        probs = torch.softmax(self.router(flat_cond), dim=-1)
        top1 = probs.argmax(dim=-1)
        if self.track_routing:
            self.routing_log.append(top1.detach().cpu())

        weights = probs.gather(1, top1.unsqueeze(-1))
        out = None
        for e in range(self.num_experts):
            mask = top1 == e
            if not mask.any():
                continue
            y = self.experts[e](flat[mask]) * weights[mask]
            if out is None:
                out = torch.zeros(flat.shape, dtype=y.dtype, device=y.device)
            out[mask] = y
        if out is None:
            out = torch.zeros_like(flat)

        return out.view(shape)

    def stored_params(self):
        return sum(p.numel() for p in self.experts.parameters())

    def active_params(self):
        return sum(p.numel() for p in self.experts[0].parameters())

    def n_options(self):
        return self.num_experts

    def configurations(self):
        return float(self.num_experts)

### Mohammad — AAG chunked expert bank

In [ ]:
class ChunkLinearMoE(nn.Module):
    """A Linear whose output rows are split into independently routed bands."""

    def __init__(self, in_features, out_features, num_chunks, num_options,
                 pretrained_weight=None, pretrained_bias=None, preserve_init=True):
        super().__init__()
        assert out_features % num_chunks == 0
        self.in_features = in_features
        self.out_features = out_features
        self.num_chunks = num_chunks
        self.num_options = num_options
        self.chunk_dim = out_features // num_chunks
        self.preserve_init = preserve_init

        self.router = nn.Linear(in_features, num_chunks * num_options)
        self.chunk_weights = nn.Parameter(
            torch.zeros(num_chunks, num_options, self.chunk_dim, in_features))
        self.chunk_biases = nn.Parameter(
            torch.zeros(num_chunks, num_options, self.chunk_dim))

        self.routing_log = []
        self.track_routing = False

    def forward(self, x):
        n = x.shape[0]
        logits = self.router(x).view(n, self.num_chunks, self.num_options)
        probs = F.softmax(logits, dim=-1)
        top_weights, top_indices = probs.max(dim=-1)
        if self.track_routing:
            self.routing_log.append(top_indices.detach().cpu())

        # Fused dispatch: all options in one matmul, then gather the winner.
        all_options = F.linear(
            x,
            self.chunk_weights.reshape(-1, self.in_features),
            self.chunk_biases.reshape(-1),
        ).view(n, self.num_chunks, self.num_options, self.chunk_dim)

        selection = top_indices[:, :, None, None].expand(
            n, self.num_chunks, 1, self.chunk_dim)
        out = all_options.gather(2, selection).squeeze(2)

        gate = top_weights.unsqueeze(-1)
        if self.preserve_init:
            gate = gate / gate.detach().clamp_min(1e-9)
        return (out * gate).reshape(n, self.out_features)


class AAGLayer(nn.Module):
    def __init__(self, original_mlp, num_chunks=8, num_options=4):
        super().__init__()
        self.num_chunks = num_chunks
        self.num_options = num_options
        hidden_dim = original_mlp.c_fc.weight.shape[0]
        intermediate_dim = original_mlp.c_fc.weight.shape[1]
        self.c_fc = ChunkLinearMoE(hidden_dim, intermediate_dim, num_chunks, num_options)
        self.act = ACT2FN['gelu_new']
        self.c_proj = ChunkLinearMoE(intermediate_dim, hidden_dim, num_chunks, num_options)

    @property
    def track_routing(self):
        return self.c_fc.track_routing

    @track_routing.setter
    def track_routing(self, value):
        self.c_fc.track_routing = value

    @property
    def routing_log(self):
        return self.c_fc.routing_log

    @routing_log.setter
    def routing_log(self, value):
        self.c_fc.routing_log = value
        self.c_proj.routing_log = []

    def forward(self, hidden_states, *args, **kwargs):
        shape = hidden_states.shape
        flat = hidden_states.view(-1, shape[-1])
        return self.c_proj(self.act(self.c_fc(flat))).view(shape)

    def stored_params(self):
        return (self.c_fc.chunk_weights.numel() + self.c_fc.chunk_biases.numel()
                + self.c_proj.chunk_weights.numel() + self.c_proj.chunk_biases.numel())

    def active_params(self):
        return (self.c_fc.out_features * self.c_fc.in_features + self.c_fc.out_features
                + self.c_proj.out_features * self.c_proj.in_features
                + self.c_proj.out_features)

    def n_options(self):
        return self.num_options

    def configurations(self):
        return (float(self.num_options) ** self.num_chunks) ** 2

### Builder and strict loader

`strict=False` is required because each variant has a different key set from
stock GPT-2, so anything silently unmatched would stay at its random
initialisation. The check below turns that silence into a hard failure.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token


def build(name):
    spec = MODELS[name]
    model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')
    model.config.pad_token_id = tokenizer.eos_token_id
    model.config.use_cache = False

    kind = spec['kind']
    for block in model.transformer.h:
        if kind == 'moe':
            block.mlp = GPT2MoELayer(block.mlp, routing=spec['routing'])
        elif kind == 'moe-cond':
            block.mlp = GPT2MoECondLayer(block.mlp)
        elif kind == 'aag':
            block.mlp = AAGLayer(block.mlp, spec['num_chunks'], spec['num_options'])

    if spec['ckpt']:
        state = load_file(f"{spec['ckpt']}/model.safetensors")
        result = model.load_state_dict(state, strict=False)

        # GPT-2 ties lm_head.weight to transformer.wte.weight -- they are the
        # same tensor -- so save_pretrained stores it once. That key is
        # legitimately absent from every checkpoint and must not count as a
        # failure. Loading wte.weight already updates lm_head through the tie.
        tied = set(getattr(model, '_tied_weights_keys', None) or [])
        if not tied and getattr(model.config, 'tie_word_embeddings', False):
            tied = {'lm_head.weight'}
        model.tie_weights()

        missing = [k for k in result.missing_keys if k not in tied]
        total = len(model.state_dict())
        if missing:
            raise RuntimeError(
                f'{name}: {len(missing)}/{total} keys did not load. The '
                f'architecture does not match the checkpoint. First missing: '
                f'{missing[:3]}')
        print(f'{name:12s} loaded {total - len(result.missing_keys)}/{total} keys'
              f' (+{len(result.missing_keys)} tied),'
              f' {len(result.unexpected_keys)} unexpected')
    else:
        print(f'{name:12s} pretrained GPT-2')

    return model.to(device).eval()


# Verify every checkpoint loads before running anything expensive.
for name in MODELS:
    m = build(name)
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Evaluation data

### Why two eval sets

The four models did not all train on the same data. Alpaca-cleaned is a
corrected version of Alpaca with a different row count, so a seeded split
produces **different held-out rows** -- some examples held out for one model
were training data for another. Any single Alpaca-derived eval set therefore
flatters two of the four.

So both are reported:

| Set | What it measures | Caveat |
|---|---|---|
| **alpaca-cleaned held-out** | in-domain quality | overlaps the training data of the two alpaca-trained models |
| **dolly-15k held-out** | generalisation | **no model trained on it** -- fully neutral |

The dolly numbers are the fair comparison. The alpaca numbers are the
familiar ones. Reporting both, and saying why, is more defensible than
picking one and hoping nobody asks.

In [ ]:
PROMPT_INPUT = ('Below is an instruction that describes a task, paired with an '
                'input that provides further context. Write a response that '
                'appropriately completes the request.')
PROMPT_PLAIN = ('Below is an instruction that describes a task. Write a response '
                'that appropriately completes the request.')
NL = chr(10)


def format_prompt(instruction, input_text=''):
    if input_text:
        return (PROMPT_INPUT + NL + NL + '### Instruction:' + NL + instruction
                + NL + NL + '### Input:' + NL + input_text + NL + NL
                + '### Response:' + NL)
    return (PROMPT_PLAIN + NL + NL + '### Instruction:' + NL + instruction
            + NL + NL + '### Response:' + NL)


def tokenize(examples):
    input_ids, attention_mask, labels = [], [], []
    for instruction, input_text, output in zip(
            examples['instruction'], examples['input'], examples['output']):
        prompt = format_prompt(instruction, input_text)
        full = prompt + output + tokenizer.eos_token
        p_ids = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)['input_ids']
        f_ids = tokenizer(full, truncation=True, max_length=MAX_LENGTH)['input_ids']
        label = [-100] * len(p_ids) + f_ids[len(p_ids):]
        pad = MAX_LENGTH - len(f_ids)
        input_ids.append(f_ids + [tokenizer.pad_token_id] * pad)
        attention_mask.append([1] * len(f_ids) + [0] * pad)
        labels.append(label + [-100] * pad)
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}


def collate(features):
    keys = ('input_ids', 'attention_mask', 'labels')
    return {k: torch.tensor([f[k] for f in features], dtype=torch.long) for k in keys}


def build_eval(name):
    if name == 'alpaca-cleaned':
        ds = load_dataset('yahma/alpaca-cleaned')['train']
        ds = ds.train_test_split(test_size=0.05, seed=SEED)['test']
    else:  # dolly -- neutral, nobody trained on it
        ds = load_dataset('databricks/databricks-dolly-15k')['train']
        ds = ds.rename_columns({'context': 'input', 'response': 'output'})
        ds = ds.train_test_split(test_size=0.10, seed=SEED)['test']
    ds = ds.select(range(min(N_EVAL, len(ds))))
    ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
    ds = ds.filter(lambda ex: any(t != -100 for t in ex['labels']))
    return ds


EVAL_SETS = {name: build_eval(name) for name in ('alpaca-cleaned', 'dolly-15k')}
for name, ds in EVAL_SETS.items():
    print(f'{name:16s} {len(ds):5d} examples')

## 5. Metrics

**Perplexity** is token-weighted over supervised tokens only. The prompt is
masked out with `-100`, so this measures response quality, not the model's
ability to echo the template. Averaging per-batch losses instead (as the
training notebooks do) over-weights short batches.

**Routing entropy / max** is 1.0 when every expert gets equal use and 0.0 when
the router has collapsed onto one. **CV** is the same story inverted.

**Active parameters per token** counts only what a single token actually
touches. This is the sparse-versus-dense distinction, and it is what makes
soft routing's cost visible.

In [ ]:
def routed_layers(model):
    for i, block in enumerate(model.transformer.h):
        if hasattr(block.mlp, 'n_options'):
            yield i, block.mlp


@torch.no_grad()
def evaluate(model, dataset, batch_size=EVAL_BATCH):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate)

    for _, mlp in routed_layers(model):
        mlp.routing_log = []
        mlp.track_routing = True

    total_nll, total_tokens = 0.0, 0
    counts = {}

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch['input_ids'],
                       attention_mask=batch['attention_mask']).logits
        shift_logits = logits[:, :-1, :].reshape(-1, logits.size(-1))
        shift_labels = batch['labels'][:, 1:].reshape(-1)
        total_nll += F.cross_entropy(shift_logits.float(), shift_labels,
                                     ignore_index=-100, reduction='sum').item()
        total_tokens += int((shift_labels != -100).sum())

        keep = batch['attention_mask'].reshape(-1).bool().cpu()
        for i, mlp in routed_layers(model):
            logs = mlp.routing_log
            if logs:
                sel = torch.cat([t.reshape(t.shape[0], -1) for t in logs], dim=0)
                if sel.shape[0] == keep.shape[0]:
                    sel = sel[keep]
                c = torch.bincount(sel.reshape(-1), minlength=mlp.n_options()).float()
                counts[i] = counts.get(i, torch.zeros(mlp.n_options())) + c
            mlp.routing_log = []

    for _, mlp in routed_layers(model):
        mlp.track_routing = False

    nll = total_nll / max(total_tokens, 1)
    return {'eval_tokens': total_tokens, 'loss': nll,
            'perplexity': math.exp(nll) if nll < 30 else float('inf'),
            'routing_counts': counts}


def routing_health(counts):
    if not counts:
        return {'per_layer': {}, 'mean_cv': None, 'entropy_ratio': None}
    per_layer, cvs, ratios = {}, [], []
    for i in sorted(counts):
        share = (counts[i] / counts[i].sum().clamp_min(1)).numpy()
        nz = share[share > 0]
        entropy = float(-(nz * np.log2(nz)).sum())
        max_entropy = math.log2(len(share))
        cv = float(np.std(share) / (np.mean(share) + 1e-8))
        per_layer[i] = {'share': share.tolist(), 'cv': cv,
                        'entropy_ratio': entropy / max_entropy}
        cvs.append(cv)
        ratios.append(entropy / max_entropy)
    return {'per_layer': per_layer, 'mean_cv': float(np.mean(cvs)),
            'entropy_ratio': float(np.mean(ratios))}


def capacity(model):
    total = sum(p.numel() for p in model.parameters())
    stored = active = 0
    configs = 1.0
    n = 0
    for _, mlp in routed_layers(model):
        stored += mlp.stored_params()
        active += mlp.active_params()
        configs *= mlp.configurations()
        n += 1
    if n == 0:
        for block in model.transformer.h:
            k = sum(p.numel() for p in block.mlp.parameters())
            stored += k
            active += k
    return {'total_params': total, 'mlp_stored': stored, 'mlp_active': active,
            'log10_configs': math.log10(configs) if configs > 0 else 0.0}


@torch.no_grad()
def benchmark_speed(model, prompt='List three healthy snacks.'):
    model.config.use_cache = True
    inputs = tokenizer(format_prompt(prompt), return_tensors='pt').to(device)
    model.generate(**inputs, max_new_tokens=8, pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=GEN_TOKENS, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    model.config.use_cache = False
    generated = out.shape[1] - inputs.input_ids.shape[1]
    return {'tokens_per_s': generated / elapsed,
            'ms_per_token': elapsed / generated * 1000,
            'peak_mb': (torch.cuda.max_memory_allocated(device) / 1024 ** 2
                        if torch.cuda.is_available() else None)}

## 6. Qualitative probes

The same prompts, the same decoding settings and the same seed for every
model, so differences in output are differences in the model.

In [ ]:
PROBE_CATEGORIES = {
    'Code': ['Write a Python function to reverse a string.',
             'How do I parse JSON in JavaScript?',
             'Explain what a for loop does.'],
    'Math': ['Solve for x: 3x + 15 = 45.',
             'What is 17 percent of 250?',
             'If all A are B and all B are C, are all A C?'],
    'Grammar': ['Correct the grammar: He do not have no money.',
                'Rephrase more professionally: I want to quit.',
                'Fix the spelling: yesturday.'],
    'Creative': ['Write a short poem about a rainy afternoon.',
                 'Draft an opening line for a science fiction novel.',
                 'Suggest a name for a coffee shop.'],
    'Factual': ['What is the capital of Canada?',
                'Who proposed the theory of general relativity?',
                'Which planet is the largest?'],
}

SHOWCASE = ['What is the capital of Canada?',
            'List three healthy snacks.',
            'Correct the grammar: He do not have no money.',
            'Write a short poem about a rainy afternoon.']


@torch.no_grad()
def generate(model, instruction, max_new_tokens=GEN_TOKENS, seed=SEED):
    model.config.use_cache = True
    torch.manual_seed(seed)
    inputs = tokenizer(format_prompt(instruction), return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         top_k=40, top_p=0.9, temperature=0.6,
                         pad_token_id=tokenizer.eos_token_id)
    model.config.use_cache = False
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                            skip_special_tokens=True).strip()


@torch.no_grad()
def category_routing(model, target_layer=0):
    """Expert usage per task category -- does routing track the task at all?"""
    mlp = model.transformer.h[target_layer].mlp
    if not hasattr(mlp, 'n_options'):
        return None
    model.config.use_cache = True
    table = {}
    for category, prompts in PROBE_CATEGORIES.items():
        totals = np.zeros(mlp.n_options())
        for prompt in prompts:
            inputs = tokenizer(format_prompt(prompt), return_tensors='pt').to(device)
            mlp.routing_log = []
            mlp.track_routing = True
            model.generate(**inputs, max_new_tokens=30, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
            mlp.track_routing = False
            if mlp.routing_log:
                sel = torch.cat([t.reshape(-1) for t in mlp.routing_log])
                totals += torch.bincount(sel, minlength=mlp.n_options()).float().numpy()
        table[category] = (100 * totals / totals.sum()).tolist() if totals.sum() else None
    model.config.use_cache = False
    return table


def specialisation_score(table):
    """Mutual information between task category and expert choice, in bits.

    0 means routing is independent of the task -- the experts are
    interchangeable. The maximum is log2(number of categories).
    """
    if not table or any(v is None for v in table.values()):
        return None
    joint = np.array([table[c] for c in table]) / 100.0
    joint = joint / joint.sum()
    p_cat = joint.sum(axis=1, keepdims=True)
    p_exp = joint.sum(axis=0, keepdims=True)
    nz = joint > 0
    return float((joint[nz] * np.log2(joint[nz] / (p_cat @ p_exp)[nz])).sum())

## 7. Run the benchmark

Each model is built, scored on both eval sets, profiled, probed, then freed.
Results are written to Drive after every model, so an interruption keeps
whatever finished.

In [ ]:
results = {}

for name, spec in MODELS.items():
    print()
    print('=' * 68)
    print(f"{spec['label']}   [{name}]   owner: {spec['owner']}")
    print('=' * 68)

    model = build(name)
    record = {'label': spec['label'], 'owner': spec['owner'],
              'trained_on': spec['trained_on'], 'capacity': capacity(model),
              'eval': {}}

    for set_name, dataset in EVAL_SETS.items():
        m = evaluate(model, dataset)
        health = routing_health(m.pop('routing_counts'))
        record['eval'][set_name] = {**m, 'routing': health}
        print(f"  {set_name:16s} perplexity {m['perplexity']:8.3f}   "
              f"entropy {health['entropy_ratio']}")

    record['speed'] = benchmark_speed(model)
    record['samples'] = {p: generate(model, p) for p in SHOWCASE}
    record['category_routing'] = category_routing(model, target_layer=0)
    record['specialisation_bits'] = specialisation_score(record['category_routing'])

    print(f"  throughput {record['speed']['tokens_per_s']:.1f} tok/s   "
          f"specialisation {record['specialisation_bits']}")

    results[name] = record
    with open(f'{OUT}/results.json', 'w') as fh:
        json.dump(results, fh, indent=2)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print(f'done -- {len(results)} models -> {OUT}/results.json')

## 8. Quantitative results

In [ ]:
rows = []
dense_active = results['gpt2-dense']['capacity']['mlp_active']

for name, r in results.items():
    cap = r['capacity']
    rows.append({
        'Model': r['label'],
        'Owner': r['owner'],
        'Trained on': r['trained_on'],
        'PPL (alpaca)': round(r['eval']['alpaca-cleaned']['perplexity'], 2),
        'PPL (dolly)': round(r['eval']['dolly-15k']['perplexity'], 2),
        'Stored MLP': f"{cap['mlp_stored'] / 1e6:.1f}M",
        'Active/token': f"{cap['mlp_active'] / 1e6:.1f}M",
        'Compute': f"{cap['mlp_active'] / dense_active:.2f}x",
        'Configs': f"1e{cap['log10_configs']:.1f}",
        'Routing H': (round(r['eval']['dolly-15k']['routing']['entropy_ratio'], 3)
                      if r['eval']['dolly-15k']['routing']['entropy_ratio'] else '--'),
        'tok/s': round(r['speed']['tokens_per_s'], 1),
    })

table = pd.DataFrame(rows)
table.to_csv(f'{OUT}/results.csv', index=False)
with open(f'{OUT}/results.tex', 'w') as fh:
    fh.write(table.to_latex(index=False, escape=True))

print(table.to_markdown(index=False))
table

### Reading the table

- **Compute** is per-token MLP arithmetic relative to vanilla GPT-2. Soft
  routing runs every expert on every token, so it pays 4x for the same four
  choices a top-1 router gets for 1x.
- **Configs** is the number of distinct weight configurations the whole model
  can assemble, log10. A 4-expert MoE over 12 layers gives 4^12; a chunked
  bank gives (4^chunks)^2 per layer.
- **Routing H** near 1.0 means every expert stays in use; near 0 means the
  router collapsed and the capacity exists on paper only.
- Prefer the **dolly** column when comparing across owners -- no model trained
  on it, so it is the only column free of the train/test overlap.

## 9. Figures

In [ ]:
SURFACE, INK, INK_SOFT, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#dcdcd8'
COLOR = {'reference': '#2a78d6', 'Tamara': '#eb6834', 'Mohammad': '#1baf7a'}


def style(ax, xlabel=None, ylabel=None, title=None):
    ax.set_facecolor(SURFACE)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_SOFT, labelsize=9)
    if xlabel:
        ax.set_xlabel(xlabel, color=INK_SOFT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=INK_SOFT, fontsize=10)
    if title:
        ax.set_title(title, color=INK, fontsize=12, fontweight='bold', loc='left', pad=12)


# --- perplexity on both eval sets, side by side ---
names = list(results)
labels = [results[n]['label'] for n in names]
colors = [COLOR[results[n]['owner']] for n in names]
y = np.arange(len(names))

fig, axes = plt.subplots(1, 2, figsize=(13, 0.6 * len(names) + 2.4), dpi=200,
                         sharey=True)
fig.patch.set_facecolor(SURFACE)

for ax, set_name, title in zip(
        axes, ['alpaca-cleaned', 'dolly-15k'],
        ['In-domain (alpaca-cleaned)', 'Neutral held-out (dolly-15k)']):
    vals = [results[n]['eval'][set_name]['perplexity'] for n in names]
    ax.barh(y, vals, height=0.6, color=colors, zorder=3)
    span = max(vals)
    for yi, v in zip(y, vals):
        ax.text(v + span * 0.015, yi, f'{v:.1f}', va='center', color=INK,
                fontsize=9, fontweight='bold')
    ax.set_xlim(0, span * 1.18)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    style(ax, xlabel='Perplexity (lower is better)', title=title)

axes[0].set_yticks(y)
axes[0].set_yticklabels(labels, color=INK, fontsize=10)
axes[0].invert_yaxis()
fig.tight_layout()
fig.savefig(f'{OUT}/fig_perplexity.png', facecolor=SURFACE)
plt.show()

In [ ]:
# --- capacity bought per unit of compute ---
fig, ax = plt.subplots(figsize=(9, 5.2), dpi=200)
fig.patch.set_facecolor(SURFACE)

xs = [results[n]['capacity']['mlp_active'] / dense_active for n in names]
ys = [results[n]['capacity']['log10_configs'] for n in names]
mid = (min(xs) + max(xs)) / 2

for n, x, yv in zip(names, xs, ys):
    ax.scatter([x], [yv], s=150, color=COLOR[results[n]['owner']],
               edgecolors=SURFACE, linewidths=2, zorder=3)
    right = x > mid
    ax.annotate(results[n]['label'], (x, yv), textcoords='offset points',
                xytext=(-12 if right else 12, 7),
                ha='right' if right else 'left', color=INK, fontsize=9)

ax.set_xlim(min(xs) - 0.3, max(xs) + 0.6)
ax.set_ylim(-max(ys) * 0.08, max(ys) * 1.25)
ax.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
style(ax, xlabel='MLP arithmetic per token, relative to vanilla GPT-2',
      ylabel='Weight configurations (log10)',
      title='Routing capacity bought per unit of compute')
fig.tight_layout()
fig.savefig(f'{OUT}/fig_capacity.png', facecolor=SURFACE)
plt.show()

In [ ]:
# --- routing health per layer, on the neutral set ---
fig, ax = plt.subplots(figsize=(9, 4.8), dpi=200)
fig.patch.set_facecolor(SURFACE)

for n in names:
    per_layer = results[n]['eval']['dolly-15k']['routing']['per_layer']
    if not per_layer:
        continue
    layers = sorted(int(k) for k in per_layer)
    vals = [per_layer[str(i)]['entropy_ratio'] if str(i) in per_layer
            else per_layer[i]['entropy_ratio'] for i in layers]
    ax.plot(layers, vals, linewidth=2, marker='o', markersize=5,
            markeredgecolor=SURFACE, markeredgewidth=1.5,
            color=COLOR[results[n]['owner']], label=results[n]['label'], zorder=3)

ax.axhline(1.0, color=GRID, linewidth=1.2, linestyle='--')
ax.text(0, 1.015, 'perfectly balanced', color=INK_SOFT, fontsize=8)
ax.set_ylim(0, 1.12)
ax.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
legend = ax.legend(frameon=False, fontsize=9, loc='lower right')
for t in legend.get_texts():
    t.set_color(INK_SOFT)
style(ax, xlabel='Transformer block', ylabel='Routing entropy / maximum',
      title='Are the experts staying in use, or collapsing?')
fig.tight_layout()
fig.savefig(f'{OUT}/fig_routing.png', facecolor=SURFACE)
plt.show()

In [ ]:
# --- expert usage by task category, one panel per routed model ---
routed = [n for n in names if results[n]['category_routing']]
if routed:
    fig, axes = plt.subplots(1, len(routed), figsize=(4.4 * len(routed), 4.2),
                             dpi=200, squeeze=False)
    fig.patch.set_facecolor(SURFACE)

    for ax, n in zip(axes[0], routed):
        table_c = results[n]['category_routing']
        cats = list(table_c)
        data = np.array([table_c[c] for c in cats])
        im = ax.imshow(data, cmap='YlGnBu', vmin=0, vmax=100, aspect='auto')
        ax.set_xticks(range(data.shape[1]))
        ax.set_xticklabels([f'e{j}' for j in range(data.shape[1])], fontsize=8)
        ax.set_yticks(range(len(cats)))
        ax.set_yticklabels(cats, fontsize=8)
        for r_i in range(data.shape[0]):
            for c_i in range(data.shape[1]):
                ax.text(c_i, r_i, f'{data[r_i, c_i]:.0f}', ha='center', va='center',
                        fontsize=7,
                        color='white' if data[r_i, c_i] > 55 else INK)
        bits = results[n]['specialisation_bits']
        ax.set_title(f"{results[n]['label']}" + chr(10) +
                     f'specialisation {bits:.3f} bits',
                     color=INK, fontsize=9, fontweight='bold')

    fig.suptitle('Expert usage by task category (layer 0)', color=INK,
                 fontsize=12, fontweight='bold', x=0.02, ha='left')
    fig.tight_layout()
    fig.savefig(f'{OUT}/fig_specialisation.png', facecolor=SURFACE)
    plt.show()

## 10. Qualitative comparison

Same prompts, same decoding settings, same seed for every model.

In [ ]:
for prompt in SHOWCASE:
    print('=' * 78)
    print('PROMPT:', prompt)
    print('=' * 78)
    for n in names:
        print()
        print(f"--- {results[n]['label']} ({results[n]['owner']})")
        print(results[n]['samples'][prompt])
    print()

In [ ]:
# Markdown table of generations, for pasting into the report
lines = ['| Model | ' + ' | '.join(SHOWCASE) + ' |',
         '|' + '---|' * (len(SHOWCASE) + 1)]
for n in names:
    cells = [results[n]['samples'][p].replace(chr(10), ' ')[:120] for p in SHOWCASE]
    lines.append(f"| {results[n]['label']} | " + ' | '.join(cells) + ' |')

generations_md = chr(10).join(lines)
with open(f'{OUT}/generations.md', 'w') as fh:
    fh.write(generations_md)
print(generations_md)

## 11b. Per-chunk routing — the measurement that actually tests AAG

The routing table above pools option usage across all 8 chunks. That is a
**marginal** distribution, and it cannot distinguish a healthy chunked router
from a fully collapsed one.

Concretely: if chunk 0 always picked option 0, chunk 1 always option 1, and
so on, then every chunk is 100% collapsed -- yet pooled across chunks the
usage is a perfect 25/25/25/25 and the entropy reads **1.000**.

So the pooled number is not wrong, it answers the wrong question. What
matters for AAG is whether **each chunk's router** is making a real choice.
That is measured below, per (layer, chunk).

Note the training objective already targets this: the auxiliary loss computes
`balance_term` per chunk and averages, so per-chunk balance is what was
optimised for. This cell checks whether that actually worked rather than
assuming it.

In [ ]:
@torch.no_grad()
def per_chunk_routing(model, dataset, batch_size=EVAL_BATCH, max_batches=25):
    """Option usage per (layer, chunk), not pooled across chunks."""
    layers = [(i, b.mlp) for i, b in enumerate(model.transformer.h)
              if hasattr(b.mlp, 'c_fc')]
    if not layers:
        return None

    n_chunks = layers[0][1].c_fc.num_chunks
    n_opts = layers[0][1].c_fc.num_options
    counts = {i: torch.zeros(n_chunks, n_opts) for i, _ in layers}

    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate)
    for n, batch in enumerate(loader):
        if n >= max_batches:
            break
        for _, mlp in layers:
            mlp.routing_log = []
            mlp.track_routing = True
        model(input_ids=batch['input_ids'].to(device),
              attention_mask=batch['attention_mask'].to(device))
        keep = batch['attention_mask'].reshape(-1).bool()
        for i, mlp in layers:
            mlp.track_routing = False
            logs = mlp.routing_log
            if not logs:
                continue
            sel = torch.cat(logs, dim=0)          # [tokens, chunks]
            if sel.shape[0] == keep.shape[0]:
                sel = sel[keep]
            for c in range(n_chunks):
                counts[i][c] += torch.bincount(sel[:, c],
                                               minlength=n_opts).float()
    return counts


def entropy_ratio(share):
    nz = share[share > 0]
    if nz.numel() == 0:
        return 0.0
    return float(-(nz * nz.log2()).sum()) / math.log2(len(share))

### Per (layer, chunk) usage, and the honest entropy

In [ ]:
aag_model = build('aag')
chunk_counts = per_chunk_routing(aag_model, EVAL_SETS['dolly-15k'])

n_opts = aag_model.transformer.h[0].mlp.c_fc.num_options
header = 'Layer | Chunk | ' + ' | '.join(f'opt {o}' for o in range(n_opts)) + ' | H/Hmax'
print(header)
print('-' * len(header))

per_chunk_H, pooled_H = [], []
for layer in sorted(chunk_counts):
    mat = chunk_counts[layer]
    for c in range(mat.shape[0]):
        share = mat[c] / mat[c].sum().clamp_min(1)
        h = entropy_ratio(share)
        per_chunk_H.append(h)
        if layer < 2:   # print the first two layers in full
            cells = ' | '.join(f'{100 * s:5.1f}%' for s in share)
            print(f'L{layer:02d}   | C{c:02d}   | {cells} | {h:.3f}')
    pooled = mat.sum(dim=0)
    pooled_H.append(entropy_ratio(pooled / pooled.sum().clamp_min(1)))
    if layer == 1:
        print('...')

print()
print(f'pooled-over-chunks entropy (what section 8 reported): {sum(pooled_H) / len(pooled_H):.4f}')
print(f'TRUE per-chunk entropy      (the meaningful number)  : {sum(per_chunk_H) / len(per_chunk_H):.4f}')
print()
collapsed = sum(1 for h in per_chunk_H if h < 0.5)
print(f'chunks with entropy below 0.5 (collapsing): {collapsed}/{len(per_chunk_H)}')
print()
if sum(per_chunk_H) / len(per_chunk_H) > 0.9:
    print('=> every chunk router is genuinely choosing between its 4 options;'
          ' the combinatorial capacity is real, not nominal.')
else:
    print('=> chunk routers have collapsed. The pooled number was hiding it,'
          ' and the configuration count is nominal only.')

del aag_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 11. Summary

In [ ]:
print('FILES WRITTEN')
for f in sorted(os.listdir(OUT)):
    print(f'   {OUT}/{f}')

print()
print('HEADLINE NUMBERS  (neutral dolly-15k held-out)')
best = min(names, key=lambda n: results[n]['eval']['dolly-15k']['perplexity'])
for n in names:
    r = results[n]
    mark = '  <-- best' if n == best else ''
    print(f"   {r['label']:34s} "
          f"ppl {r['eval']['dolly-15k']['perplexity']:8.2f}   "
          f"compute {r['capacity']['mlp_active'] / dense_active:.2f}x   "
          f"configs 1e{r['capacity']['log10_configs']:.1f}{mark}")